In [2]:
# Lab 4.2 Image Generation with Stable Diffusion
!pip install diffusers transformers accelerate

import torch
from diffusers import StableDiffusionPipeline
from PIL import Image

# CONFIG
MODEL_ID = "runwayml/stable-diffusion-v1-5" # or "stabilityai/stable-diffusion-2-1-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256 # Use 512 for better quality if GPU allows

# LOAD PIPELINE
print("Loading Stable Diffusion pipeline...")
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    safety_checker=None # disable safety checker for faster inference
)
pipe = pipe.to(DEVICE)

# Optional: enable memory-efficient attention (xformers)
# pipe.enable_xformers_memory_efficient_attention()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading Stable Diffusion pipeline...


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [3]:
# GENERATE IMAGES
prompts = [
    "A serene mountain landscape at sunset, photorealistic, 4k",
    "A futuristic city with flying cars, neon lights, cyberpunk style",
    "A cat wearing a spacesuit floating in space, digital art"
]

for i, prompt in enumerate(prompts):
    print(f"\nGenerating image {i+1}/{len(prompts)}: {prompt}")
    image = pipe(
        prompt=prompt,
        negative_prompt="blurry, low quality, distorted",
        height=IMG_SIZE,
        width=IMG_SIZE,
        num_inference_steps=30, # 20-50 typical; more = higher quality but slower
        guidance_scale=7.5, # 7-15 typical; higher = stronger prompt adherence
        num_images_per_prompt=1,
        generator=torch.manual_seed(42), # for reproducibility
    ).images[0]

    image.save(f"output_{i+1}.png")
    print(f"Saved output_{i+1}.png")


Generating image 1/3: A serene mountain landscape at sunset, photorealistic, 4k


  0%|          | 0/30 [00:00<?, ?it/s]

Saved output_1.png

Generating image 2/3: A futuristic city with flying cars, neon lights, cyberpunk style


  0%|          | 0/30 [00:00<?, ?it/s]

Saved output_2.png

Generating image 3/3: A cat wearing a spacesuit floating in space, digital art


  0%|          | 0/30 [00:00<?, ?it/s]

Saved output_3.png


In [4]:
# EXPERIMENT WITH PARAMETERS
prompt = "A dragon breathing fire, fantasy art"
print("\n--- Experimenting with guidance scale ---")

guidance_scales = [15.0]
for gs in guidance_scales:
    image = pipe(
        prompt=prompt,
        height=IMG_SIZE,
        width=IMG_SIZE,
        num_inference_steps=25,
        guidance_scale=gs,
        generator=torch.manual_seed(42),
    ).images[0]

    image.save(f"dragon_gs_{gs}.png")
    print(f"Saved dragon_gs_{gs}.png (guidance scale {gs})")

# ====
# 5. INPAINTING (OPTIONAL)
# ====
# Requires StableDiffusionInpaintPipeline
# from diffusers import StableDiffusionInpaintPipeline
# inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
#     "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16
# ).to(DEVICE)
#
# # Load image and mask (white = inpaint region)
# image = Image.open("photo.png").resize((512, 512))
# mask = Image.open("mask.png").resize((512, 512))
#
# result = inpaint_pipe(
#     prompt="a red sofa",
#     image=image,
#     mask_image=mask,
#     num_inference_steps=30,
# ).images[0]
# result.save("inpainted.png")

print("\nLab 4.2 complete. Check the output images.")


--- Experimenting with guidance scale ---


  0%|          | 0/25 [00:00<?, ?it/s]

Saved dragon_gs_15.0.png (guidance scale 15.0)

Lab 4.2 complete. Check the output images.
